In [ ]:
!pip install torch datasets huggingface_hub umap-learn accelerate scikit-learn matplotlib seaborn transformers optuna shap nltk wordcloud

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk
from transformers import (AutoTokenizer, AutoModel, Trainer, TrainingArguments,
                          DataCollatorWithPadding, set_seed, EarlyStoppingCallback,
                          BertForSequenceClassification)
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from huggingface_hub import login
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix

import umap
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
import re
from scipy.stats import pointbiserialr
from nltk.corpus import stopwords
import nltk
import shap
import string

In [ ]:
login("...") #Here, we need to input the HuggingFace Token

In [ ]:
MODEL_NAME = "mental/mental-bert-base-uncased"
DATASET_PATH = "daic_woz_symptom_dataset"
MAX_LEN = 64
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
ds = load_from_disk(DATASET_PATH)

def tokenize_segment(examples):
    tokens = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)
    tokens["labels"] = [l[:8] + [float(d)] for l, d in zip(examples["labels"], examples["diagnosis"])]
    return tokens

tokenized_ds = ds.map(tokenize_segment, batched=True, remove_columns=["text"])
tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "patient_id"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
ds

In [ ]:
ds["test"][4]

In [ ]:
symptom_names = ["Interest", "Mood", "Sleep", "Energy", "Appetite", "Self-Esteem", "Concentration", "Psychomotor"]

true_labels = np.array(tokenized_ds["test"]["labels"])[:, :8]

for i, sym in enumerate(symptom_names):
    mean_val = true_labels[:, i].mean()
    zero_percentage = (true_labels[:, i] == 0).mean() * 100
    print(f"{sym:<15} | Mean: {mean_val:.2f} | Percentage of 0s: {zero_percentage:.1f}%")

In [ ]:
class SMART(nn.Module):
    def __init__(self, model_name, hidden_dim=128, dropout_prob=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.queries = nn.Parameter(torch.randn(9, 768))
        
        def create_head(output_dim):
            return nn.Sequential(
                nn.Linear(768, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout_prob),
                nn.Linear(hidden_dim, output_dim)
            )

        self.symptom_heads = nn.ModuleList([create_head(1) for _ in range(8)])
        self.diagnosis_head = create_head(1)

    def forward(self, input_ids, attention_mask, labels=None):
        states = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        scores = torch.matmul(states, self.queries.T)
        scores = scores.masked_fill(attention_mask.unsqueeze(2) == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        feats = torch.matmul(weights.transpose(1, 2), states)
        
        s_preds = torch.cat([torch.sigmoid(self.symptom_heads[i](feats[:, i, :])) * 3.0 for i in range(8)], dim=1)
        d_logits = self.diagnosis_head(feats[:, 8, :])
        
        return s_preds, d_logits

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred.predictions, eval_pred.label_ids
    pred_diagnosis = (preds[:, :8].sum(axis=1) >= 10.0).astype(int)
    return {"f1": f1_score(labels[:, 8], pred_diagnosis, average="macro")}

In [ ]:
class MentalBertTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        s_preds, d_logits = model(**inputs)

        mse = F.mse_loss(s_preds, labels[:, :8])
        bce = F.binary_cross_entropy_with_logits(d_logits, labels[:, 8].view(-1, 1), pos_weight=torch.tensor([3.0], device=d_logits.device))

        centered = s_preds - s_preds.mean(dim=0, keepdim=True)
        norm = centered / (centered.std(dim=0, keepdim=True) + 1e-8)
        corr = torch.matmul(norm.T, norm) / (norm.shape[0] - 1)
        mask = ~torch.eye(8, dtype=bool, device=corr.device)
        corr_penalty = F.relu(corr[mask].abs() - 0.7).mean()    
        loss = (0.7 * mse) + (0.3 * bce) + (0.3 * corr_penalty)
        
        return (loss, {"logits": torch.cat([s_preds, d_logits], dim=1)}) if return_outputs else loss

In [ ]:
model = SMART(MODEL_NAME)

training_args = TrainingArguments(
    output_dir="./SMART",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer = MentalBertTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

In [ ]:
trainer.train()

In [ ]:
model.eval()
model.to(device)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def get_patient_scores(dataset):
    loader = torch.utils.data.DataLoader(
        dataset.remove_columns(["patient_id"]), 
        batch_size=16, 
        collate_fn=collator
    )
    
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device) 
            scores, _ = model(input_ids, mask)
            all_preds.append(scores.cpu().numpy())
            all_labels.append(batch["labels"][:, 8].cpu().numpy())
    
    preds_np = np.concatenate(all_preds).sum(axis=1)
    labels_np = np.concatenate(all_labels)
    
    df = pd.DataFrame({
        "pid": dataset["patient_id"], 
        "pred": preds_np, 
        "true": labels_np
    })
    return df.groupby("pid").mean()

val_df = get_patient_scores(tokenized_ds["validation"])
mean_healthy = val_df[val_df["true"] < 0.5]["pred"].mean()
mean_depressed = val_df[val_df["true"] >= 0.5]["pred"].mean()
threshold_shift = 10.0 - ((mean_healthy + mean_depressed) / 2)
test_df = get_patient_scores(tokenized_ds["test"])

y_true = test_df["true"].astype(int)
y_pred = ((test_df["pred"] + threshold_shift) >= 10.0).astype(int)

print(classification_report(y_true, y_pred, target_names=["Healthy", "Depressed"]))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=False, xticklabels=["Healthy", "Depressed"], yticklabels=["Healthy", "Depressed"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
#plt.savefig("confusion_matrix.pdf", format="pdf", bbox_inches="tight")
#plt.close()
plt.show()

In [ ]:
def get_detailed_symptom_errors(eval_model, test_ds, eval_tokenizer):
    eval_model.eval()
    eval_model.to(device)
    collator = DataCollatorWithPadding(tokenizer=eval_tokenizer)
    
    loader = torch.utils.data.DataLoader(test_ds.remove_columns(["patient_id"]), batch_size=16, collate_fn=collator)
    
    all_preds, all_trues = [], []
    with torch.no_grad():
        for b in loader:
            s_preds, _ = eval_model(b["input_ids"].to(device), b["attention_mask"].to(device))
            all_preds.append(s_preds.cpu().numpy())
            all_trues.append(b["labels"][:, :8].cpu().numpy())
            
    preds_np = np.concatenate(all_preds)
    trues_np = np.concatenate(all_trues)

    df_list = []
    for i, sym in enumerate(symptom_names):
        temp_df = pd.DataFrame({
            "pid": test_ds["patient_id"],
            "symptom": sym,
            "pred": preds_np[:, i],
            "true": trues_np[:, i]
        })
        df_list.append(temp_df)
        
    full_df = pd.concat(df_list)
    patient_df = full_df.groupby(["pid", "symptom"]).mean().reset_index()
    patient_df["absolute_error"] = (patient_df["pred"] - patient_df["true"]).abs()
    return patient_df

error_df = get_detailed_symptom_errors(model, tokenized_ds["test"], tokenizer)

print("Error Sleep")
sleep_errors = error_df[error_df["symptom"] == "Sleep"].sort_values(by="absolute_error", ascending=False)
print(sleep_errors.head(3))

print("Error Energy")
energy_errors = error_df[error_df["symptom"] == "Energy"].sort_values(by="absolute_error", ascending=False)
print(energy_errors.head(3))

In [ ]:
class SMARTSymptomOnly(nn.Module):
    def __init__(self, model_name, hidden_dim=128, dropout_prob=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.queries = nn.Parameter(torch.randn(8, 768))
        
        def create_head(output_dim):
            return nn.Sequential(
                nn.Linear(768, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout_prob),
                nn.Linear(hidden_dim, output_dim)
            )

        self.symptom_heads = nn.ModuleList([create_head(1) for _ in range(8)])

    def forward(self, input_ids, attention_mask, labels=None):
        states = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state   
        scores = torch.matmul(states, self.queries.T)
        scores = scores.masked_fill(attention_mask.unsqueeze(2) == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        feats = torch.matmul(weights.transpose(1, 2), states)
        s_preds = torch.cat([torch.sigmoid(self.symptom_heads[i](feats[:, i, :])) * 3.0 for i in range(8)], dim=1)

        return s_preds

In [ ]:
def compute_metrics_symptom_only(eval_pred):
    preds, labels = eval_pred.predictions, eval_pred.label_ids
    pred_diagnosis = (preds.sum(axis=1) >= 10.0).astype(int)
    return {"f1": f1_score(labels[:, 8], pred_diagnosis, average="macro")}

class SMARTSymptomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        s_preds = model(**inputs)

        mse = F.mse_loss(s_preds, labels[:, :8])

        centered = s_preds - s_preds.mean(dim=0, keepdim=True)
        norm = centered / (centered.std(dim=0, keepdim=True) + 1e-8)
        corr = torch.matmul(norm.T, norm) / (norm.shape[0] - 1)
        mask = ~torch.eye(8, dtype=bool, device=corr.device)
        corr_penalty = F.relu(corr[mask].abs() - 0.7).mean()
        loss = mse + (0.3 * corr_penalty)
        
        return (loss, {"logits": s_preds}) if return_outputs else loss

In [ ]:
ablation_model = SMARTSymptomOnly(MODEL_NAME)

ablation_args = TrainingArguments(
    output_dir="./mental_bert_symptom_only",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

ablation_trainer = SMARTSymptomTrainer(
    model=ablation_model,
    args=ablation_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_symptom_only,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

In [ ]:
ablation_trainer.train()

In [ ]:
ablation_model.eval()
ablation_model.to(device)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def get_patient_scores_ablation(dataset):
    loader = torch.utils.data.DataLoader(
        dataset.remove_columns(["patient_id"]), 
        batch_size=16, 
        collate_fn=collator
    )
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            scores = ablation_model(input_ids, mask)
            all_preds.append(scores.cpu().numpy())
            all_labels.append(batch["labels"][:, 8].cpu().numpy())

    preds_np = np.concatenate(all_preds).sum(axis=1)
    labels_np = np.concatenate(all_labels)
    
    df = pd.DataFrame({
        "pid": dataset["patient_id"], 
        "pred": preds_np, 
        "true": labels_np
    })
    return df.groupby("pid").mean()

val_df = get_patient_scores_ablation(tokenized_ds["validation"])
mean_healthy = val_df[val_df["true"] < 0.5]["pred"].mean()
mean_depressed = val_df[val_df["true"] >= 0.5]["pred"].mean()
threshold_shift = 10.0 - ((mean_healthy + mean_depressed) / 2)
test_df = get_patient_scores_ablation(tokenized_ds["test"])

y_true = test_df["true"].astype(int)
y_pred = ((test_df["pred"] + threshold_shift) >= 10.0).astype(int)

print(classification_report(y_true, y_pred, target_names=["Healthy", "Depressed"]))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=False, xticklabels=["Healthy", "Depressed"], yticklabels=["Healthy", "Depressed"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
#plt.savefig("confusion_matrix_ablation.pdf", format="pdf", bbox_inches="tight")
#plt.close()
plt.show()

# untrained BERT baseline

In [ ]:
def get_binary_diagnosis(batch):
    batch["labels"] = [int(label_vector[8]) for label_vector in batch["labels"]]
    return batch

test_ds_binary = tokenized_ds["test"].map(get_binary_diagnosis, batched=True)
baseline_model_untrained = BertForSequenceClassification.from_pretrained("mental/mental-bert-base-uncased", num_labels=2).to(device)

eval_args = TrainingArguments(
    output_dir="./baseline_untrained", 
    per_device_eval_batch_size=16, 
    remove_unused_columns=True
)

baseline_trainer = Trainer(
    model=baseline_model_untrained, 
    args=eval_args, 
    data_collator=data_collator
)

pred_results = baseline_trainer.predict(test_ds_binary)
segment_preds = np.argmax(pred_results.predictions, axis=1)
segment_labels = pred_results.label_ids

df_segments = pd.DataFrame({
    "patient_id": test_ds_binary["patient_id"][:len(segment_preds)],
    "pred": segment_preds,
    "true_label": segment_labels
})

def get_majority_vote(series):
    return series.mode().iloc[0]
    
df_patients = df_segments.groupby("patient_id").agg({
    "pred": get_majority_vote, 
    "true_label": "first"
})

y_true = df_patients["true_label"]
y_pred = df_patients["pred"]

print(classification_report(y_true, y_pred, target_names=["Healthy", "Depressed"], zero_division=0))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=False,xticklabels=["Healthy", "Depressed"],yticklabels=["Healthy", "Depressed"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
#plt.savefig("confusion_matrix_untrained.pdf", format="pdf", bbox_inches="tight")
#plt.close()
plt.show()

# trained BERT baseline


In [ ]:
train_ds_binary = tokenized_ds["train"].map(get_binary_diagnosis, batched=True)
val_ds_binary = tokenized_ds["validation"].map(get_binary_diagnosis, batched=True)

baseline_model_trained = BertForSequenceClassification.from_pretrained(
    "mental/mental-bert-base-uncased", 
    num_labels=2
).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        class_weights = torch.tensor([1.0, 3.0], device=model.device) #war 2.1
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        logits = outputs.get("logits")
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

train_args = TrainingArguments(
    output_dir="./baseline_trained", 
    num_train_epochs=3, 
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16, 
    learning_rate=2e-5, 
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch", 
    load_best_model_at_end=True, 
    report_to="none"
)

baseline_trainer = WeightedTrainer(
    model=baseline_model_trained,
    args=train_args,
    train_dataset=train_ds_binary,
    eval_dataset=val_ds_binary,
    data_collator=data_collator
)

baseline_trainer.train()

pred_results_trained = baseline_trainer.predict(test_ds_binary)
segment_preds = np.argmax(pred_results_trained.predictions, axis=1)
segment_labels = pred_results_trained.label_ids

df_segments = pd.DataFrame({
    "patient_id": test_ds_binary["patient_id"][:len(segment_preds)],
    "pred": segment_preds,
    "true_label": segment_labels
})

df_patients = df_segments.groupby("patient_id").agg({
    "pred": get_majority_vote, 
    "true_label": "first"
})

y_true = df_patients["true_label"]
y_pred = df_patients["pred"]

print(classification_report(y_true, y_pred, target_names=["Healthy", "Depressed"]))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=False,xticklabels=["Healthy", "Depressed"],yticklabels=["Healthy", "Depressed"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
#plt.savefig("confusion_matrix_trained.pdf", format="pdf", bbox_inches="tight")
#plt.close()
plt.show()

# correlation matrix

In [ ]:
model.eval()

eval_loader = torch.utils.data.DataLoader(
    tokenized_ds["test"].remove_columns(["patient_id"]), 
    batch_size=16, 
    collate_fn=data_collator
)

all_symptom_preds = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        
        batch_scores, _ = model(input_ids, mask)
        all_symptom_preds.append(batch_scores.cpu().numpy())

preds_array = np.concatenate(all_symptom_preds, axis=0)
df_preds = pd.DataFrame(preds_array, columns=symptom_names)
df_preds["patient_id"] = tokenized_ds["test"]["patient_id"]

patient_means = df_preds.groupby("patient_id").mean()
corr_matrix = patient_means.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix.iloc[1:, :-1], mask=mask[1:, :-1], annot=True, fmt=".2f", cmap="Purples", vmin=0.0, vmax=1.0, linewidths=.5, cbar_kws={"shrink": .8})
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
#plt.savefig("symptom_correlation.pdf", format="pdf", bbox_inches="tight")
#plt.close()
plt.show()

In [ ]:
train_labels = np.array(tokenized_ds["train"]["labels"])
df_true = pd.DataFrame(train_labels[:, :8], columns=symptom_names)
df_true["patient_id"] = tokenized_ds["train"]["patient_id"]

patient_true_means = df_true.groupby("patient_id").mean()
true_corr_matrix = patient_true_means.corr()

mask = np.triu(np.ones_like(true_corr_matrix, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(true_corr_matrix.iloc[1:, :-1], mask=mask[1:, :-1], annot=True, fmt=".2f", cmap="Purples", vmin=0.0, vmax=1.0, linewidths=.5, cbar_kws={"shrink": .8})
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
#plt.savefig("symptom_correlation_true.pdf", format="pdf", bbox_inches="tight")
#plt.close()
plt.show()

In [ ]:
model.eval()

eval_loader = DataLoader(
    tokenized_ds["test"].remove_columns(["patient_id"]), 
    batch_size=16, 
    collate_fn=data_collator
)

all_s_preds, all_s_true = [], []

with torch.no_grad():
    for batch in eval_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        s_preds, _ = model(ids, mask)
        all_s_preds.append(s_preds.cpu().numpy())
        all_s_true.append(batch["labels"][:, :8].cpu().numpy())

preds_array = np.concatenate(all_s_preds, axis=0)
true_array = np.concatenate(all_s_true, axis=0)

df_eval = pd.DataFrame({"patient_id": tokenized_ds["test"]["patient_id"]})

for i, name in enumerate(symptom_names):
    df_eval[f"pred_{name}"] = preds_array[:, i]
    df_eval[f"true_{name}"] = true_array[:, i]

agg_rules = {f"pred_{name}": "mean" for name in symptom_names}
agg_rules.update({f"true_{name}": "first" for name in symptom_names})

patient_level_df = df_eval.groupby("patient_id").agg(agg_rules).reset_index()

metrics_data = []
for name in symptom_names:
    y_true = patient_level_df[f"true_{name}"]
    y_pred = patient_level_df[f"pred_{name}"]
    metrics_data.append({"Symptom": name,"MAE": mean_absolute_error(y_true, y_pred),"MSE": mean_squared_error(y_true, y_pred)})
results_df = pd.DataFrame(metrics_data)

print(results_df.round(4).to_string(index=False))
print(f"Mean MAE: {results_df['MAE'].mean():.4f}, Mean MSE: {results_df['MSE'].mean():.4f}\n")

mae_plot = results_df["MAE"].tolist() + [results_df["MAE"].iloc[0]]
mse_plot = results_df["MSE"].tolist() + [results_df["MSE"].iloc[0]]

angles = np.linspace(0, 2 * np.pi, len(symptom_names), endpoint=False).tolist()
angles += angles[:1]  # Close the angle loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, mae_plot, color='#1f77b4', linewidth=2, linestyle='solid', label='MAE')
ax.fill(angles, mae_plot, color='#1f77b4', alpha=0.25)
ax.plot(angles, mse_plot, color='#d62728', linewidth=2, linestyle='dashed', label='MSE')
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(symptom_names, fontsize=11, weight='bold')
max_err = max(max(mae_plot), max(mse_plot))
ax.set_ylim(0, max_err + 0.1)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
#plt.savefig("radar_chart_errors.pdf", format="pdf", bbox_inches="tight")
#plt.close()
plt.show()

In [ ]:
def plot_umap(model, dataset, title, model_type):
    model.eval()
    device = next(model.parameters()).device
    
    loader = DataLoader(
        dataset.remove_columns(["patient_id"]), 
        batch_size=16, 
        collate_fn=data_collator
    )

    all_scores, all_cls_tokens, all_labels = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=mask)
            
            if model_type == "multitask":
                batch_scores = outputs[0].cpu().numpy().sum(axis=1)
            else:
                batch_scores = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()

            backbone_outputs = model.bert(input_ids=input_ids, attention_mask=mask)
            cls_token = backbone_outputs.last_hidden_state[:, 0, :].cpu().numpy()

            all_scores.append(batch_scores)
            all_cls_tokens.append(cls_token)
            labels = batch["labels"].cpu().numpy()
            if labels.ndim > 1:
                all_labels.append(labels[:, 8])
            else:
                all_labels.append(labels)

    df_data = {
        "patient_id": dataset["patient_id"],
        "score": np.concatenate(all_scores),
        "true_diagnosis": np.concatenate(all_labels)
    }
    
    cls_matrix = np.concatenate(all_cls_tokens)
    for i in range(cls_matrix.shape[1]): # 768 dimensions
        df_data[f"dim_{i}"] = cls_matrix[:, i]
        
    df_segments = pd.DataFrame(df_data)
    patient_df = df_segments.groupby("patient_id").mean()
    cls_features = patient_df[[f"dim_{i}" for i in range(768)]].values
    scaled_features = StandardScaler().fit_transform(cls_features)
    reducer = umap.UMAP(n_neighbors=10, min_dist=0.3, random_state=42)
    embeddings_2d = reducer.fit_transform(scaled_features)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    diagnosis_labels = ["Depressed" if label >= 0.5 else "Healthy" for label in patient_df["true_diagnosis"]]
    sns.scatterplot(x=embeddings_2d[:, 0], y=embeddings_2d[:, 1], hue=diagnosis_labels, palette={'Healthy': '#2ecc71', 'Depressed': '#e74c3c'}, s=100, ax=ax1)
    ax1.set_title("True Diagnosis", fontsize=14, fontweight='bold')

    sc = ax2.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=patient_df["score"], cmap='YlOrRd', s=100)
    cbar_label = 'Severity Score' if model_type == "multitask" else 'Prediction Probability'
    ax2_title = 'Predicted Severity' if model_type == "multitask" else 'Prediction Probability'
    cbar = fig.colorbar(sc, ax=ax2)
    cbar.set_label(cbar_label, fontsize=12)
    ax2.set_title(ax2_title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    title = title.replace(' ', '_').lower()
    #plt.savefig(f"umap_{title}.pdf", format="pdf", bbox_inches="tight")
    #plt.close()
    plt.show()

In [ ]:
plot_umap(baseline_model_untrained, tokenized_ds["test"], "Untrained MentalBERT", "binary")

In [ ]:
plot_umap(baseline_model_trained, tokenized_ds["test"], "Trained Baseline BERT", "binary")

In [ ]:
plot_umap(model, tokenized_ds["test"], "SMART", "multitask")

In [ ]:
import string
import numpy as np
import pandas as pd
import torch
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import nltk
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english')).union({
    'know', 'think', 'like', 'yeah', 'yes', 'no', 'well', 'um', 'uh', 
    'really', 'just', 'get', 'got', 'mean', 'thing', 'something', 
    'would', 'could', 'going', 'say', 'said', 'im', 'dont', 'thats', 'right', 'lot', 'little', 'much'
})
alias_map = {"pts": "ptsd"}

def predict_symptom_scores(texts):
    if isinstance(texts, np.ndarray): 
        texts = texts.tolist()    
    encoded = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    model_inputs = {k: v for k, v in encoded.items() if k in ["input_ids", "attention_mask"]}
    with torch.no_grad():
        outputs = model(**model_inputs)

    return outputs[0][:, :8].cpu().numpy()

def aggregate_word_impact(word, shap_values, sums_dict, counts_dict):
    clean_word = word.lower().translate(str.maketrans('', '', string.punctuation))
    clean_word = alias_map.get(clean_word, clean_word)
    if clean_word and clean_word not in stop_words and len(clean_word) > 2 and not clean_word.isnumeric():
        positive_impact = np.maximum(shap_values, 0)
        sums_dict[clean_word] += positive_impact
        counts_dict[clean_word] += 1

def compute_global_shap(texts, min_frequency=3):
    def get_zero_array():
        return np.zeros(len(symptom_names))

    explainer = shap.Explainer(predict_symptom_scores, tokenizer)
    shap_results = explainer(texts) 
    word_shap_sums = defaultdict(get_zero_array)
    word_counts = defaultdict(int)
    
    for text_explanation in shap_results:
        current_word = ""
        current_vals = get_zero_array()
        for token, shap_val in zip(text_explanation.data, text_explanation.values):
            token_str = token.strip()
            if not token_str: 
                continue
            if token_str.startswith('##'):
                current_word += token_str.replace('##', '')
                current_vals += shap_val
            else:
                if current_word:
                    aggregate_word_impact(current_word, current_vals, word_shap_sums, word_counts) 
                current_word = token_str.replace('Ġ', '')
                current_vals = np.copy(shap_val)     
        if current_word:
            aggregate_word_impact(current_word, current_vals, word_shap_sums, word_counts)
    mean_importance = {word: word_shap_sums[word] / count for word, count in word_counts.items() if count >= min_frequency}
            
    return pd.DataFrame.from_dict(mean_importance, orient='index', columns=symptom_names)

def plot_top_shap(importance_df, top_n=8):
    plt.rcParams['font.family'] = 'serif'
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    for i, symptom in enumerate(symptom_names):
        ax = axes.flatten()[i]
        if symptom in importance_df.columns:
            top_words = importance_df[symptom].sort_values(ascending=False).head(top_n)
            sns.barplot(x=top_words.values, y=top_words.index, ax=ax, palette="viridis")
            ax.set_title(symptom, fontsize=14, fontweight='bold')
            ax.set_xlabel("Mean Positive SHAP Value", fontsize=11)
            ax.set_ylabel("")
            ax.tick_params(axis='y', labelsize=12)
            sns.despine(ax=ax)
            
    plt.tight_layout()
    #plt.savefig("shap_global.pdf", format="pdf", bbox_inches="tight")
    #plt.close()
    plt.show()

importance_df = compute_global_shap(ds["test"]["text"], min_frequency=3)
plot_top_shap(importance_df, top_n=8)

In [ ]:
def plot_tired_distribution(dataset):
    texts = dataset["text"]
    labels = np.array(dataset["labels"]) 
    tired_mask = np.array([bool(re.search(r'\btired\b', str(t).lower())) for t in texts])
        
    tired_labels = labels[tired_mask][:, :8]
    other_labels = labels[~tired_mask][:, :8]
    
    tired_means = tired_labels.mean(axis=0)
    other_means = other_labels.mean(axis=0)
    
    df_plot = pd.DataFrame({
        "Symptom": symptom_names * 2,
        "Mean True Score": np.concatenate([tired_means, other_means]),
        "Context": ['Contains "tired"'] * 8 + ['Does not contain "tired"'] * 8
    })
    
    plt.rcParams['font.family'] = 'serif'
    plt.figure(figsize=(10, 5))
    palette = {'Contains "tired"': "#1f77b4", 'Does not contain "tired"': "#ff7f0e"} 
    ax = sns.barplot(data=df_plot, x="Symptom", y="Mean True Score", hue="Context", palette=palette)
    
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', padding=3, fontsize=9)
        
    ax.set_ylim(0, df_plot["Mean True Score"].max() * 1.15)
    plt.xlabel("PHQ-8 Symptoms", fontsize=12)
    plt.ylabel("Mean True Severity Score", fontsize=12)
    plt.xticks(rotation=45, ha="right", fontsize=11)
    plt.legend(frameon=False, fontsize=11, title="")
    sns.despine()
    plt.tight_layout()
    #plt.savefig("tired_word_distribution.pdf", format="pdf", bbox_inches="tight")
    #plt.close()
    plt.show()
    
plot_tired_distribution(ds["test"])

In [ ]:
sleep_texts = [t for t in ds["test"]["text"] if "sleep" in t.lower() or "wake" in t.lower()]
mot_texts = [t for t in ds["test"]["text"] if "motivated" in t.lower() or "energy" in t.lower()]
mood_texts = [t for t in ds["test"]["text"] if "sad" in t.lower() or "depress" in t.lower()]
negation_texts = [
    t for t in ds["test"]["text"] 
    if ("not" in t.lower()) and ("depressed" in t.lower())
]

In [ ]:
def raw_logit_predict(texts):
    if isinstance(texts, np.ndarray): texts = texts.tolist()
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = model(**{k: v for k, v in enc.items() if k in ["input_ids", "attention_mask"]})
        logits = out[0] if isinstance(out, (tuple, list)) else (getattr(out, 'logits', out))
    return logits[:, :8].cpu().numpy()

def analyze_real_sentence_thesis(text, name):
    
    explainer = shap.Explainer(raw_logit_predict, tokenizer)
    shap_values = explainer([text])
    tokens = shap_values.data[0]
    valid_indices, clean_tokens = [], []
    
    for i, t in enumerate(tokens):
        t_clean = t.replace('##', '').replace('Ġ', '').strip()
        if len(t_clean) > 0 and t_clean not in ["[CLS]", "[SEP]", "[PAD]"]:
            valid_indices.append(i)
            clean_tokens.append(t_clean)
            
    matrix = shap_values.values[0][valid_indices, :]
    
    fig_height = max(2.5, len(clean_tokens) * 0.5)
    plt.figure(figsize=(10, fig_height))
    plt.rcParams['font.family'] = 'serif'
    sns.heatmap(matrix, annot=True, fmt=".2f", xticklabels=symptom_names, yticklabels=clean_tokens, 
                cmap="RdBu_r", center=0, cbar_kws={'label': 'SHAP Value'},
                linewidths=0.5, linecolor='white')
    
    plt.xlabel("PHQ-8 Symptoms", fontsize=12, labelpad=10)
    plt.ylabel("") 
    
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)
    
    plt.tight_layout()
    #plt.savefig(filename, format="pdf", bbox_inches="tight", dpi=300)
    #plt.close()
    plt.show()

analyze_real_sentence_thesis(sleep_texts[0],"shap_local_sleep.pdf")
analyze_real_sentence_thesis(mot_texts[len(mot_texts)-1], "shap_local_motivation.pdf")
analyze_real_sentence_thesis(negation_texts[0], "shap_local_negation.pdf")
analyze_real_sentence_thesis("tired", "shap_perturbation_tired.pdf")
    

In [ ]:
def probe_layers(target_model, dataset, target_vals):
    target_model.eval()
    cols = ["input_ids", "attention_mask"]
    subset = dataset.select(range(len(target_vals)))
    subset = subset.remove_columns([c for c in subset.column_names if c not in cols])
    loader = DataLoader(subset, batch_size=8, collate_fn=data_collator)
    layer_representations = {i: [] for i in range(13)}
    
    with torch.no_grad():
        for b in loader:
            inputs = {k: v.to(device) for k, v in b.items()}
            bert = target_model.bert if hasattr(target_model, 'bert') else target_model
            out = bert(**inputs, output_hidden_states=True)
            for l, state in enumerate(out.hidden_states):
                layer_representations[l].append(state.mean(dim=1).cpu().numpy())

    results = []

    null_preds = np.full_like(target_vals, fill_value=np.mean(target_vals))
    base_entropy = np.log(mean_squared_error(target_vals, null_preds))
    
    for l in range(13):
        X = np.concatenate(layer_representations[l])
        probe = Ridge(alpha=1.0).fit(X, target_vals)
        layer_entropy = np.log(mean_squared_error(target_vals, probe.predict(X)))
        info_gain = base_entropy - layer_entropy
        results.append(info_gain)
        
    return results

In [ ]:
def get_surprisal_labels(text_list):
    model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model.eval()
    results = []
    for text in text_list:
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
        logits = outputs.logits[..., :-1, :].contiguous()
        labels = inputs["input_ids"][..., 1:].contiguous()
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), reduction="none")
        results.append(loss.mean().item())
    return np.array(results)

def get_pronoun_density_labels(text_list):
    first_person_singular = {'i', 'me', 'my', 'mine', 'myself'}
    results = []
    for text in text_list:
        words = re.findall(r'\w+', text.lower())
        if not words:
            densities.append(0.0)
            continue
        count = sum(1 for w in words if w in first_person_singular)
        results.append(count / len(words))
    return np.array(results)

def get_absolutist_density_labels(text_list):
    absolutist_words = {
        'always', 'never', 'completely', 'every', 'entirely', 'total', 
        'none', 'all', 'must', 'constantly', 'definitely', 'absolutely'
    }
    results = []
    for text in text_list:
        words = re.findall(r'\w+', text.lower())
        if not words:
            densities.append(0.0)
            continue
        count = sum(1 for word in words if word in absolutist_words)
        results.append(count / len(words))
    return np.array(results)

In [ ]:
proxies = []
proxies.append(get_surprisal_labels(ds["test"]["text"]))
proxies.append(get_pronoun_density_labels(ds["test"]["text"]))
proxies.append(get_absolutist_density_labels(ds["test"]["text"]))

for targets in proxies:
    results = {
        "SMART": probe_layers(model, tokenized_ds["test"], targets),
        "Baseline (Trained)": probe_layers(baseline_model_trained, tokenized_ds["test"], targets),
        "Baseline (Untrained)": probe_layers(baseline_model_untrained, tokenized_ds["test"], targets)
    }
    
    plt.rcParams['font.family'] = 'serif'
    plt.figure(figsize=(8, 5))
    colors = {"SMART": "#1f77b4", "Baseline (Trained)": "#2ca02c", "Baseline (Untrained)": "#ff7f0e"}
    styles = {"SMART": "-", "Baseline (Trained)": "--", "Baseline (Untrained)": ":"}
    markers = {"SMART": "o", "Baseline (Trained)": "s", "Baseline (Untrained)": "^"}
    linewidths = {"SMART": 2.0, "Baseline (Trained)": 1.5, "Baseline (Untrained)": 1.5}
    for name, scores in results.items():
        plt.plot(range(13), scores, 
                 marker=markers.get(name, "o"), 
                 markersize=6, 
                 color=colors.get(name, "#333333"), 
                 linestyle=styles.get(name, "-"), 
                 linewidth=linewidths.get(name, 1.5), 
                 label=name)
    
    plt.xlabel("Layer", fontsize=12)
    plt.xticks(range(13), ["0 (Emb)"] + [str(i) for i in range(1, 13)])
    plt.ylabel("Information Gain", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    sns.despine()
    plt.legend(frameon=False, fontsize=11)
    plt.tight_layout()
    #plt.savefig("layer_probing_absolutist_new.pdf", format="pdf", bbox_inches="tight")
    #plt.close()
    plt.show()